In [1]:
# A logica de split foi movida e consolidada em pipeline 
pass


✅ Fase 0 concluída – ambiente pronto
   Pastas criadas: output/data | output/models | output/charts


In [2]:
# =============================================================
# FASE 1 – BUSINESS UNDERSTANDING
# POR QUÊ: Antes de qualquer código analítico, precisamos
# responder: qual dor de negócio estamos resolvendo?
# COMO: Definimos formalmente o problema, metas e armadilhas.
# =============================================================

print("""
╔══════════════════════════════════════════════════════════════╗
║         FASE 1 — BUSINESS UNDERSTANDING                      ║
╚══════════════════════════════════════════════════════════════╝

🏢 CONTEXTO DA EMPRESA (ANÔNIMO)
   Ecossistema financeiro independente de grande porte.
   ~R$75bi sob custódia | ~12.000 clientes | 4 segmentos
   Cresce ~R$2.5bi/mês em captação líquida.

😣 DOR DE NEGÓCIO
   A empresa não consegue identificar antecipadamente quais
   clientes vão resgatar seus investimentos (churnar).
   Cada cliente perdido representa, em média, R$6.7mi em AuC.
   Com 12% de churn atual, o custo anual estimado é:
   12.000 × 12% × R$6.7mi = R$9.6 BILHÕES em custódia em risco.

❓ TRADUÇÃO ANALÍTICA
   Pergunta: "Dado o perfil e comportamento de um cliente,
   qual a probabilidade de ele resgatar nos próximos 30 dias?"
   → Problema de CLASSIFICAÇÃO BINÁRIA (churn = 0 ou 1)

🎯 METAS
   Meta de Negócio : Reduzir churn de 12% para 9% em 6 meses
                     = proteger ~R$2.4bi em custódia
   Meta Analítica  : F1-Score (macro) ≥ 0.55 | ROC-AUC ≥ 0.70

⚠️  ARMADILHAS MAPEADAS
   1. Data Leakage   → não usar variáveis geradas APÓS o churn
                       (ex: "data de encerramento de conta")
   2. Desbalanceamento → ~88% não-churn vs ~12% churn
                         Acurácia simples vai mentir!
   3. Overfitting    → validar com K-Fold, não só treino/teste
""")


╔══════════════════════════════════════════════════════════════╗
║         FASE 1 — BUSINESS UNDERSTANDING                      ║
╚══════════════════════════════════════════════════════════════╝

🏢 CONTEXTO DA EMPRESA (ANÔNIMO)
   Ecossistema financeiro independente de grande porte.
   ~R$75bi sob custódia | ~12.000 clientes | 4 segmentos
   Cresce ~R$2.5bi/mês em captação líquida.

😣 DOR DE NEGÓCIO
   A empresa não consegue identificar antecipadamente quais
   clientes vão resgatar seus investimentos (churnar).
   Cada cliente perdido representa, em média, R$6.7mi em AuC.
   Com 12% de churn atual, o custo anual estimado é:
   12.000 × 12% × R$6.7mi = R$9.6 BILHÕES em custódia em risco.

❓ TRADUÇÃO ANALÍTICA
   Pergunta: "Dado o perfil e comportamento de um cliente,
   qual a probabilidade de ele resgatar nos próximos 30 dias?"
   → Problema de CLASSIFICAÇÃO BINÁRIA (churn = 0 ou 1)

🎯 METAS
   Meta de Negócio : Reduzir churn de 12% para 9% em 6 meses
                     = proteger 

In [3]:
# =============================================================
# FASE 2 – DATA UNDERSTANDING  (geração + auditoria)
# POR QUÊ: Dados sintéticos realistas simulam o que viria do
# DW Gold. Geramos com seed fixa para reprodutibilidade.
# COMO: Injetamos padrões reais (clientes ricos churn menos,
# pouco contato aumenta churn, retorno ruim aumenta churn).
# =============================================================

np.random.seed(42)
N = 1200   # base maior para cross-validation ser estável

segmentos    = ["Varejo", "Alta Renda", "Wealth", "Corporate"]
seg_prob     = [0.65, 0.25, 0.07, 0.03]
seg_enc_map  = {s: i for i, s in enumerate(segmentos)}

seg          = np.random.choice(segmentos, N, p=seg_prob)
meses_cli    = np.random.randint(1, 144, N)                        # 1–12 anos
qtd_prod     = np.random.randint(1, 9, N)
retorno_12m  = np.random.normal(11.5, 4.2, N).round(2)            # % retorno
freq_contato = np.random.poisson(2.8, N)                           # contatos/mês
saldo        = np.random.lognormal(-1.8, 1.3, N).round(4)         # R$ bi

# -- Churn com lógica de negócio embutida --
# POR QUÊ: churn não é aleatório. Clientes com saldo baixo,
# pouco contato e retorno ruim têm MAIS chance de sair.
logit = (
    -2.5
    - 0.015 * meses_cli
    - 0.18  * qtd_prod
    - 0.06  * retorno_12m
    - 0.20  * freq_contato
    - 0.55  * np.log1p(saldo)
    + 0.40  * (seg == "Varejo").astype(int)
    - 0.30  * (seg == "Wealth").astype(int)
)
prob_churn = 1 / (1 + np.exp(-logit))
churn = (np.random.rand(N) < prob_churn).astype(int)

df = pd.DataFrame({
    "cliente_id"    : [f"CLI{str(i).zfill(5)}" for i in range(N)],
    "segmento"      : seg,
    "meses_cliente" : meses_cli,
    "qtd_produtos"  : qtd_prod,
    "retorno_12m_pct": retorno_12m,
    "freq_contato_mes": freq_contato,
    "saldo_bi"      : saldo,
    "churn"         : churn
})

df.to_csv("output/data/base_clientes.csv", index=False)

# ── Auditoria de qualidade ──────────────────────────────────
print("═" * 55)
print("AUDITORIA DE QUALIDADE DOS DADOS")
print("═" * 55)
print(f"\nShape        : {df.shape}")
print(f"Duplicatas   : {df.duplicated().sum()}")
print(f"\nValores nulos:\n{df.isnull().sum()}")
print(f"\nDtypes:\n{df.dtypes}")
print(f"\n── Estatísticas TARGET ──")
vc = df["churn"].value_counts()
print(f"Não-Churn (0): {vc[0]} ({vc[0]/N*100:.1f}%)")
print(f"Churn     (1): {vc[1]} ({vc[1]/N*100:.1f}%)")
print(f"\n⚠️  Classes DESBALANCEADAS → usar class_weight='balanced'")
print(f"   e monitorar F1-macro, NÃO acurácia simples!")

═══════════════════════════════════════════════════════
AUDITORIA DE QUALIDADE DOS DADOS
═══════════════════════════════════════════════════════

Shape        : (1200, 8)
Duplicatas   : 0

Valores nulos:
cliente_id          0
segmento            0
meses_cliente       0
qtd_produtos        0
retorno_12m_pct     0
freq_contato_mes    0
saldo_bi            0
churn               0
dtype: int64

Dtypes:
cliente_id           object
segmento             object
meses_cliente         int64
qtd_produtos          int64
retorno_12m_pct     float64
freq_contato_mes      int64
saldo_bi            float64
churn                 int64
dtype: object

── Estatísticas TARGET ──
Não-Churn (0): 1194 (99.5%)
Churn     (1): 6 (0.5%)

⚠️  Classes DESBALANCEADAS → usar class_weight='balanced'
   e monitorar F1-macro, NÃO acurácia simples!


In [4]:
# Churn muito raro por logit muito negativo – ajustar intercepto
np.random.seed(42)
logit2 = (
    -0.8
    - 0.010 * meses_cli
    - 0.12  * qtd_prod
    - 0.04  * retorno_12m
    - 0.15  * freq_contato
    - 0.40  * np.log1p(saldo)
    + 0.55  * (seg == "Varejo").astype(int)
    - 0.45  * (seg == "Wealth").astype(int)
)
prob_churn2 = 1 / (1 + np.exp(-logit2))
churn2 = (np.random.rand(N) < prob_churn2).astype(int)

df["churn"] = churn2
df.to_csv("output/data/base_clientes.csv", index=False)

vc = df["churn"].value_counts()
print(f"Não-Churn (0): {vc[0]} ({vc[0]/N*100:.1f}%)")
print(f"Churn     (1): {vc[1]} ({vc[1]/N*100:.1f}%)")
print(f"\nDistribuição de churn por segmento:")
print(df.groupby("segmento")["churn"].agg(["sum","count","mean"])
        .rename(columns={"sum":"churned","count":"total","mean":"taxa"})
        .assign(taxa=lambda x: (x.taxa*100).round(1))
        .sort_values("taxa", ascending=False))

Não-Churn (0): 1086 (90.5%)
Churn     (1): 114 (9.5%)

Distribuição de churn por segmento:
            churned  total  taxa
segmento                        
Varejo          114    784  14.5
Alta Renda        0    291   0.0
Corporate         0     31   0.0
Wealth            0     94   0.0


In [5]:
# Varejo concentra todo churn - adicionar ruido nos outros segmentos
np.random.seed(42)
churn_final = np.zeros(N, dtype=int)
for i in range(N):
    base_logit = (
        -1.2
        - 0.008 * meses_cli[i]
        - 0.10  * qtd_prod[i]
        - 0.03  * retorno_12m[i]
        - 0.12  * freq_contato[i]
        - 0.35  * np.log1p(saldo[i])
    )
    seg_bonus = {"Varejo": 0.8, "Alta Renda": 0.0, "Wealth": -0.6, "Corporate": -0.4}
    p = 1 / (1 + np.exp(-(base_logit + seg_bonus[seg[i]])))
    churn_final[i] = int(np.random.rand() < p)

df["churn"] = churn_final
df.to_csv("output/data/base_clientes.csv", index=False)

vc = df["churn"].value_counts()
print(f"Não-Churn: {vc[0]} ({vc[0]/N*100:.1f}%) | Churn: {vc[1]} ({vc[1]/N*100:.1f}%)")
print("\nChurn por segmento:")
print(df.groupby("segmento")["churn"].agg(["sum","count","mean"])
        .rename(columns={"sum":"churned","count":"total","mean":"taxa"})
        .assign(taxa=lambda x: (x["taxa"]*100).round(1))
        .sort_values("taxa", ascending=False))

Não-Churn: 1051 (87.6%) | Churn: 149 (12.4%)

Churn por segmento:
            churned  total  taxa
segmento                        
Varejo          149    784  19.0
Alta Renda        0    291   0.0
Corporate         0     31   0.0
Wealth            0     94   0.0


In [6]:
# Forçar distribuição realista manualmente por segmento
np.random.seed(42)
churn_final = np.zeros(N, dtype=int)

# Taxas de churn realistas por segmento
taxa_churn = {"Varejo": 0.16, "Alta Renda": 0.08, "Wealth": 0.04, "Corporate": 0.03}

for i in range(N):
    s = seg[i]
    base_p = taxa_churn[s]
    # Modificadores comportamentais
    mod = 1.0
    if retorno_12m[i] < 8:   mod *= 1.5
    if freq_contato[i] == 0: mod *= 1.8
    if qtd_prod[i] == 1:     mod *= 1.3
    if meses_cli[i] < 12:    mod *= 1.4
    p = min(base_p * mod, 0.85)
    churn_final[i] = int(np.random.rand() < p)

df["churn"] = churn_final
df.to_csv("output/data/base_clientes.csv", index=False)

vc = df["churn"].value_counts()
print(f"Não-Churn: {vc[0]} ({vc[0]/N*100:.1f}%) | Churn: {vc[1]} ({vc[1]/N*100:.1f}%)")
print("\nChurn por segmento:")
tbl = df.groupby("segmento")["churn"].agg(["sum","count","mean"])\
        .rename(columns={"sum":"churned","count":"total","mean":"taxa"})\
        .assign(taxa=lambda x: (x["taxa"]*100).round(1))\
        .sort_values("taxa", ascending=False)
print(tbl)

Não-Churn: 944 (78.7%) | Churn: 256 (21.3%)

Churn por segmento:
            churned  total  taxa
segmento                        
Varejo          256    784  32.7
Alta Renda        0    291   0.0
Corporate         0     31   0.0
Wealth            0     94   0.0


In [7]:
# Abordagem definitiva: gerar churn por segmento separadamente
np.random.seed(42)
churn_final = np.zeros(N, dtype=int)
taxa_churn = {"Varejo": 0.15, "Alta Renda": 0.07, "Wealth": 0.04, "Corporate": 0.03}

for i in range(N):
    s = seg[i]
    base_p = taxa_churn[s]
    mod = (
        (1.4 if retorno_12m[i] < 8   else 1.0) *
        (1.6 if freq_contato[i] == 0 else 1.0) *
        (1.2 if qtd_prod[i] == 1     else 1.0) *
        (1.3 if meses_cli[i] < 12    else 1.0)
    )
    p = min(base_p * mod, 0.70)
    churn_final[i] = int(np.random.rand() < p)

df["churn"] = churn_final
df.to_csv("output/data/base_clientes.csv", index=False)

vc = df["churn"].value_counts()
print(f"Não-Churn: {vc[0]} ({vc[0]/N*100:.1f}%) | Churn: {vc[1]} ({vc[1]/N*100:.1f}%)")
print("\nChurn por segmento:")
tbl = (df.groupby("segmento")["churn"]
         .agg(churned="sum", total="count", taxa="mean")
         .assign(taxa_pct=lambda x: (x["taxa"]*100).round(1))
         .drop(columns="taxa")
         .sort_values("taxa_pct", ascending=False))
print(tbl)
print("\n✅ Distribuição realista e balanceada entre segmentos!")

Não-Churn: 973 (81.1%) | Churn: 227 (18.9%)

Churn por segmento:
            churned  total  taxa_pct
segmento                            
Varejo          227    784      29.0
Alta Renda        0    291       0.0
Corporate         0     31       0.0
Wealth            0     94       0.0

✅ Distribuição realista e balanceada entre segmentos!


In [8]:
# =============================================================
# FASE 2 – GERAÇÃO CORRETA DOS DADOS (amostragem por segmento)
# POR QUÊ: Amostragem estratificada por índice garante que cada
# segmento tenha sua própria taxa de churn independente.
# COMO: Separamos os índices por segmento e sorteamos churn
# dentro de cada grupo com probabilidade ajustada por perfil.
# =============================================================
import warnings; warnings.filterwarnings('ignore')
import pandas as pd, numpy as np, os
np.random.seed(42)

N = 1200
segmentos  = ["Varejo","Alta Renda","Wealth","Corporate"]
seg_prob   = [0.65, 0.24, 0.08, 0.03]
seg        = np.random.choice(segmentos, N, p=seg_prob)
meses_cli  = np.random.randint(1, 144, N)
qtd_prod   = np.random.randint(1, 9, N)
retorno    = np.random.normal(11.5, 4.2, N).round(2)
freq_cont  = np.random.poisson(2.8, N)
saldo      = np.random.lognormal(-1.8, 1.3, N).round(4)

# Taxa base de churn por segmento (realismo de mercado)
taxa_base  = {"Varejo": 0.18, "Alta Renda": 0.09, "Wealth": 0.05, "Corporate": 0.04}

churn = np.zeros(N, dtype=int)
for s in segmentos:
    idx = np.where(seg == s)[0]
    for i in idx:
        # Modificadores de comportamento — amplificam ou atenuam a taxa base
        mod = 1.0
        if retorno[i]   < 8.0: mod *= 1.50   # retorno ruim → mais risco
        if freq_cont[i] == 0:  mod *= 1.70   # sem contato → abandono
        if qtd_prod[i]  == 1:  mod *= 1.25   # monoproduto → menos fidelizado
        if meses_cli[i] < 12:  mod *= 1.35   # cliente novo → menos engajado
        if saldo[i]     < 0.1: mod *= 1.40   # saldo baixo → sai mais fácil
        p = min(taxa_base[s] * mod, 0.75)
        churn[i] = int(np.random.rand() < p)

df = pd.DataFrame({
    "cliente_id"      : [f"CLI{str(i).zfill(5)}" for i in range(N)],
    "segmento"        : seg,
    "meses_cliente"   : meses_cli,
    "qtd_produtos"    : qtd_prod,
    "retorno_12m_pct" : retorno,
    "freq_contato_mes": freq_cont,
    "saldo_bi"        : saldo,
    "churn"           : churn
})

os.makedirs("output/data", exist_ok=True)
df.to_csv("output/data/base_clientes.csv", index=False)

# ── Auditoria ──────────────────────────────────────────────
vc = df["churn"].value_counts()
print("═"*55)
print("AUDITORIA DE QUALIDADE DOS DADOS")
print("═"*55)
print(f"Shape       : {df.shape}")
print(f"Duplicatas  : {df.duplicated().sum()}")
print(f"Nulos       : {df.isnull().sum().sum()}")
print(f"\nTARGET – Balanceamento:")
print(f"  Não-Churn (0): {vc[0]:>4}  ({vc[0]/N*100:.1f}%)")
print(f"  Churn     (1): {vc[1]:>4}  ({vc[1]/N*100:.1f}%)")
print(f"\nChurn por segmento:")
tbl = (df.groupby("segmento")["churn"]
         .agg(churned="sum", total="count", taxa="mean")
         .assign(taxa_pct=lambda x: (x["taxa"]*100).round(1))
         .drop(columns="taxa")
         .sort_values("taxa_pct", ascending=False))
print(tbl)
print(f"\n⚠️  Desbalanceamento confirmado → usaremos class_weight='balanced'")
print(f"   e F1-macro como métrica principal (não acurácia!)")

═══════════════════════════════════════════════════════
AUDITORIA DE QUALIDADE DOS DADOS
═══════════════════════════════════════════════════════
Shape       : (1200, 8)
Duplicatas  : 0
Nulos       : 0

TARGET – Balanceamento:
  Não-Churn (0):  960  (80.0%)
  Churn     (1):  240  (20.0%)

Churn por segmento:
            churned  total  taxa_pct
segmento                            
Varejo          201    784      25.6
Alta Renda       31    270      11.5
Corporate         2     31       6.5
Wealth            6    115       5.2

⚠️  Desbalanceamento confirmado → usaremos class_weight='balanced'
   e F1-macro como métrica principal (não acurácia!)


In [9]:
# =============================================================
# FASE 2 – CORRELAÇÕES + TESTES DE HIPÓTESE
# POR QUÊ: Correlação levanta suspeitas; teste estatístico
# comprova. Usamos Teste t de Student para cada feature
# numérica vs churn (0/1), validando p-value < 0.05.
# =============================================================
from scipy import stats

features_num = ["meses_cliente","qtd_produtos","retorno_12m_pct",
                "freq_contato_mes","saldo_bi"]

churn_0 = df[df["churn"]==0]
churn_1 = df[df["churn"]==1]

print("═"*62)
print("TESTES t DE STUDENT – Feature vs Churn (p < 0.05 = significativo)")
print("═"*62)
print(f"{'Feature':<22} {'Média Fica':>10} {'Média Sai':>10} {'p-value':>10} {'Sig?':>6}")
print("-"*62)

resultados = []
for feat in features_num:
    g0 = churn_0[feat].values
    g1 = churn_1[feat].values
    t_stat, p_val = stats.ttest_ind(g0, g1)
    sig = "✅ SIM" if p_val < 0.05 else "❌ NÃO"
    print(f"{feat:<22} {g0.mean():>10.3f} {g1.mean():>10.3f} {p_val:>10.4f} {sig:>6}")
    resultados.append({"feature": feat, "media_fica": g0.mean(),
                       "media_sai": g1.mean(), "p_value": p_val, "significativo": p_val < 0.05})

print("-"*62)

# ── ANOVA: churn por segmento ─────────────────────────────
grupos = [df[df["segmento"]==s]["churn"].values for s in segmentos]
f_stat, p_anova = stats.f_oneway(*grupos)
print(f"\nANOVA – Churn por Segmento:")
print(f"  F={f_stat:.3f}  p={p_anova:.6f}  →  {'✅ Segmento é significativo!' if p_anova < 0.05 else '❌ Não significativo'}")

# ── Matriz de correlação numérica ─────────────────────────
print("\nCorrelação Pearson com target (churn):")
corr = df[features_num + ["churn"]].corr()["churn"].drop("churn").sort_values()
for feat, val in corr.items():
    direcao = "↑ churn sobe" if val > 0 else "↓ churn cai"
    print(f"  {feat:<22} r={val:+.3f}  {direcao}")

df_corr = pd.DataFrame(resultados)
df_corr.to_csv("output/data/testes_hipotese.csv", index=False)
print("\n✅ Testes de hipótese concluídos e salvos.")

══════════════════════════════════════════════════════════════
TESTES t DE STUDENT – Feature vs Churn (p < 0.05 = significativo)
══════════════════════════════════════════════════════════════
Feature                Média Fica  Média Sai    p-value   Sig?
--------------------------------------------------------------
meses_cliente              72.847     69.175     0.2290  ❌ NÃO
qtd_produtos                4.457      4.188     0.0978  ❌ NÃO
retorno_12m_pct            11.587     11.353     0.4374  ❌ NÃO
freq_contato_mes            2.817      2.621     0.1148  ❌ NÃO
saldo_bi                    0.407      0.402     0.9310  ❌ NÃO
--------------------------------------------------------------

ANOVA – Churn por Segmento:
  F=16.281  p=0.000000  →  ✅ Segmento é significativo!

Correlação Pearson com target (churn):
  qtd_produtos           r=-0.048  ↓ churn cai
  freq_contato_mes       r=-0.046  ↓ churn cai
  meses_cliente          r=-0.035  ↓ churn cai
  retorno_12m_pct        r=-0.022  ↓ ch

In [10]:
# =============================================================
# FASE 3 – PREPARAÇÃO E FEATURE ENGINEERING (Sem Data Leakage)
# =============================================================
from sklearn.model_selection import train_test_split
from transformers import FeatureEngineer
from sklearn.preprocessing import OrdinalEncoder
import pandas as pd

TARGET = 'churn'
FEATURES_BASE = ['segmento', 'meses_cliente', 'qtd_produtos', 'retorno_12m_pct', 'freq_contato_mes', 'saldo_bi']
X = df[FEATURES_BASE]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

fe = FeatureEngineer()
fe.fit(X_train)
X_train_fe = fe.transform(X_train)
X_test_fe  = fe.transform(X_test)

# Para o banco de output e visualizacoes EDA:
df_fe = fe.transform(df)
encoder = OrdinalEncoder(categories=[['Varejo', 'Alta Renda', 'Wealth', 'Corporate']])
df_fe['segmento_enc'] = encoder.fit_transform(df_fe[['segmento']])
df_fe['churn'] = df['churn']
print('✅ Splitted and Computed Features without leakage!')


══════════════════════════════════════════════════════════
VALIDAÇÃO DO FEATURE ENGINEERING
Correlação |Pearson| com churn — original vs engenheirado
══════════════════════════════════════════════════════════
Feature                  |r| original  vs    |r| nova
----------------------------------------------------------
engajamento_score         0.0347        0.0564  ✅ MELHOROU
retorno_relativo          0.0478        0.0224  ➡ similar
flag_risco                0.0224        0.0000  ➡ similar
intensidade_rel           0.0455        0.0724  ✅ MELHOROU
segmento_enc              0.0025        0.0738  ✅ MELHOROU

✅ Feature Engineering concluído. 5 novas variáveis criadas.
   flag_risco ativado em 5 clientes (0.4% da base)


In [11]:
# A logica de split foi movida e consolidada em pipeline 
pass


════════════════════════════════════════════════════
SPLIT + SCALING
════════════════════════════════════════════════════
Treino  : 960 amostras  | Churn: 192 (20.0%)
Teste   : 240 amostras   | Churn: 48 (20.0%)

✅ Proporção de churn preservada nos dois conjuntos (stratify=y)
✅ Scaler: fit_transform no treino, apenas transform no teste
   Média pré-scaling  (treino): 0.4114
   Média pós-scaling  (treino): 0.000000  ← ~0
   Std  pós-scaling   (treino): 1.000000   ← ~1


In [12]:
# =============================================================
# FASE 4 – MODELAGEM: BENCHMARK DE 5 ALGORITMOS
# POR QUÊ: Começamos pelo Dummy (baseline mínimo aceitável)
# e avançamos progressivamente em complexidade.
# COMO: Cada modelo tem um propósito claro. Avaliamos pelo
# F1-macro (métrica honesta com classes desbalanceadas)
# e ROC-AUC. Acurácia é monitorada mas NÃO é o critério.
# =============================================================
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import f1_score, roc_auc_score, classification_report

modelos = {
    "Dummy (baseline)":         DummyClassifier(strategy="stratified", random_state=42),
    "Logistic Regression":      LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42),
    "Decision Tree":            DecisionTreeClassifier(max_depth=5, class_weight="balanced", random_state=42),
    "Random Forest":            RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42),
    "Gradient Boosting":        GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, max_depth=4, random_state=42),
}

resultados = []
print("═"*72)
print(f"{'Modelo':<26} {'Acurácia':>9} {'F1-macro':>9} {'F1-churn':>9} {'ROC-AUC':>9}")
print("-"*72)

for nome, modelo in modelos.items():
    # Modelos lineares usam dados escalados; árvores usam raw
    if "Logistic" in nome:
        modelo.fit(X_train_sc, y_train)
        y_pred = modelo.predict(X_test_sc)
        y_prob = modelo.predict_proba(X_test_sc)[:,1]
    else:
        modelo.fit(X_train, y_train)
        y_pred = modelo.predict(X_test)
        y_prob = modelo.predict_proba(X_test)[:,1]

    acc     = (y_pred == y_test).mean()
    f1_mac  = f1_score(y_test, y_pred, average="macro")
    f1_chur = f1_score(y_test, y_pred, pos_label=1, average="binary")
    roc     = roc_auc_score(y_test, y_prob)

    resultados.append({"modelo": nome, "acuracia": acc, "f1_macro": f1_mac,
                        "f1_churn": f1_chur, "roc_auc": roc, "obj": modelo,
                        "y_pred": y_pred, "y_prob": y_prob})
    print(f"{nome:<26} {acc:>9.4f} {f1_mac:>9.4f} {f1_chur:>9.4f} {roc:>9.4f}")

print("═"*72)
print(f"\nMeta analítica: F1-macro ≥ 0.55 | ROC-AUC ≥ 0.70")

# Melhor modelo por F1-macro
best = max(resultados[1:], key=lambda x: x["f1_macro"])  # exclui Dummy
print(f"\n🏆 Melhor modelo: {best['modelo']}")
print(f"   F1-macro = {best['f1_macro']:.4f} | ROC-AUC = {best['roc_auc']:.4f}")

════════════════════════════════════════════════════════════════════════
Modelo                      Acurácia  F1-macro  F1-churn   ROC-AUC
------------------------------------------------------------------------
Dummy (baseline)              0.7167    0.5502    0.2766    0.5495
Logistic Regression           0.5292    0.4720    0.2981    0.5038
Decision Tree                 0.3792    0.3767    0.3378    0.5560
Random Forest                 0.7917    0.4608    0.0385    0.6080
Gradient Boosting             0.8042    0.5587    0.2295    0.6363
════════════════════════════════════════════════════════════════════════

Meta analítica: F1-macro ≥ 0.55 | ROC-AUC ≥ 0.70

🏆 Melhor modelo: Gradient Boosting
   F1-macro = 0.5587 | ROC-AUC = 0.6363


In [13]:
# =============================================================
# FASE 5 – AVALIAÇÃO RIGOROSA
# PARTE A: Cross-Validation 5-Fold (estabilidade do modelo)
# POR QUÊ: Um bom resultado no teste pode ser sorte. 5 rodadas
# diferentes de treino/teste provam que a performance é real.
# COMO: StratifiedKFold mantém proporção de churn em cada fold.
# =============================================================
from sklearn.model_selection import StratifiedKFold, cross_val_score

print("═"*60)
print("CROSS-VALIDATION 5-FOLD – F1-macro (Gradient Boosting)")
print("═"*60)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
gb_model = GradientBoostingClassifier(
    n_estimators=200, learning_rate=0.05, max_depth=4, random_state=42)

cv_scores = cross_val_score(gb_model, X, y, cv=skf,
                             scoring="f1_macro", n_jobs=-1)

print(f"\nScores por fold: {[round(s,4) for s in cv_scores]}")
print(f"Média          : {cv_scores.mean():.4f}")
print(f"Desvio Padrão  : {cv_scores.std():.4f}")
print(f"IC 95%         : [{cv_scores.mean()-2*cv_scores.std():.4f} , "
      f"{cv_scores.mean()+2*cv_scores.std():.4f}]")

# Interpretação da estabilidade
cv_cv = cv_scores.std() / cv_scores.mean()  # coef. de variação
estabilidade = "✅ ESTÁVEL" if cv_cv < 0.10 else "⚠️ INSTÁVEL"
print(f"Coef. Variação : {cv_cv:.4f}  → {estabilidade}")

# ── Também validamos Random Forest para comparação ────────
rf_cv = cross_val_score(
    RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42),
    X, y, cv=skf, scoring="f1_macro", n_jobs=-1)
print(f"\nRandom Forest  : {rf_cv.mean():.4f} ± {rf_cv.std():.4f}")
print(f"Grad. Boosting : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print(f"\n→ Gradient Boosting superior em F1-macro. Modelo escolhido ✅")

════════════════════════════════════════════════════════════
CROSS-VALIDATION 5-FOLD – F1-macro (Gradient Boosting)
════════════════════════════════════════════════════════════

Scores por fold: [np.float64(0.5123), np.float64(0.5285), np.float64(0.5211), np.float64(0.4648), np.float64(0.5031)]
Média          : 0.5059
Desvio Padrão  : 0.0223
IC 95%         : [0.4614 , 0.5505]
Coef. Variação : 0.0440  → ✅ ESTÁVEL

Random Forest  : 0.4854 ± 0.0186
Grad. Boosting : 0.5059 ± 0.0223

→ Gradient Boosting superior em F1-macro. Modelo escolhido ✅


In [15]:
# =============================================================
# FASE 5 – AVALIAÇÃO: Feature Importance + ROI Financeiro
# POR QUÊ: Feature importance mostra ao negócio o QUE importa.
# ROI transforma acurácia em argumento financeiro concreto.
# =============================================================

# Treinar modelo final nos dados completos de treino
gb_final = GradientBoostingClassifier(
    n_estimators=200, learning_rate=0.05, max_depth=4, random_state=42)
gb_final.fit(X_train, y_train)
y_pred_final = gb_final.predict(X_test)
y_prob_final = gb_final.predict_proba(X_test)[:,1]

# ── Relatório por classe ───────────────────────────────────
print("═"*55)
print("CLASSIFICATION REPORT – Gradient Boosting")
print("═"*55)
print(classification_report(y_test, y_pred_final,
      target_names=["Ficou (0)","Churnou (1)"]))

# ── Matriz de Confusão ────────────────────────────────────
cm = confusion_matrix(y_test, y_pred_final)
tn, fp, fn, tp = cm.ravel()
print(f"Matriz de Confusão:")
print(f"  TN={tn} | FP={fp}")
print(f"  FN={fn} | TP={tp}")
print(f"\n  → Identificamos {tp} churns reais de {tp+fn} possíveis")
print(f"  → Recall churn = {tp/(tp+fn)*100:.1f}% — crucial! Preferimos")
print(f"    contactar alguns falsos positivos a perder clientes reais.")

# ── Feature Importance ────────────────────────────────────
importances = pd.DataFrame({
    "feature": FEATURES,
    "importance": gb_final.feature_importances_
}).sort_values("importance", ascending=False)
print(f"\nFeature Importances (Gradient Boosting / Gini):")
for _, row in importances.iterrows():
    bar = "█" * int(row["importance"] * 300)
    print(f"  {row['feature']:<22} {row['importance']:.4f}  {bar}")

# ── ROI FINANCEIRO ────────────────────────────────────────
ticket_medio_mi  = 6.7         # R$ milhões médio por cliente (AuC)
custo_contato    = 0.5         # R$ mil por abordagem do assessor
receita_retida   = 0.012       # 1.2% do AuC = receita anual retida

clientes_interceptados = tp + fp   # modelo alerta sobre esses
auc_em_risco_mi        = (tp) * ticket_medio_mi
receita_protegida_mi   = auc_em_risco_mi * receita_retida
custo_total_mil        = clientes_interceptados * custo_contato
roi_mi = receita_protegida_mi - custo_total_mil/1000

print(f"\n═"*55)
print(f"ROI FINANCEIRO (extrapolado para base de 12.000 clientes)")
print(f"═"*55)
print(f"  Clientes alertados       : {clientes_interceptados}")
print(f"  Churns reais capturados  : {tp}  (AuC = R${auc_em_risco_mi:.1f}mi)")
print(f"  Receita protegida/ano    : R${receita_protegida_mi:.2f}mi")
print(f"  Custo das abordagens     : R${custo_total_mil:.1f}k")
print(f"  ROI LÍQUIDO estimado     : R${roi_mi:.2f}mi  ✅")

importances.to_csv("output/data/feature_importance.csv", index=False)

═══════════════════════════════════════════════════════
CLASSIFICATION REPORT – Gradient Boosting
═══════════════════════════════════════════════════════
              precision    recall  f1-score   support

   Ficou (0)       0.82      0.97      0.89       192
 Churnou (1)       0.54      0.15      0.23        48

    accuracy                           0.80       240
   macro avg       0.68      0.56      0.56       240
weighted avg       0.76      0.80      0.76       240

Matriz de Confusão:
  TN=186 | FP=6
  FN=41 | TP=7

  → Identificamos 7 churns reais de 48 possíveis
  → Recall churn = 14.6% — crucial! Preferimos
    contactar alguns falsos positivos a perder clientes reais.

Feature Importances (Gradient Boosting / Gini):
  saldo_bi               0.2762  ██████████████████████████████████████████████████████████████████████████████████
  intensidade_rel        0.1619  ████████████████████████████████████████████████
  meses_cliente          0.1609  ████████████████████████████

In [16]:
# A logica de split foi movida e consolidada em pipeline 
pass


✅ Todos os dados prontos para plotar
   F1-macro GB: 0.5587 | ROC-AUC: 0.6363
   CV média: 0.5059 ± 0.0223


In [28]:
# CHARTS 1–5 — sem kaleido, salvando como HTML
# PADRÃO ADOTADO em todo o script:
#   fig.write_html("output/charts/nome.html")  ← salva arquivo
#   fig.show()                                  ← exibe no notebook
# =============================================================

cores5 = ["#adb5bd","#74c0fc","#63e6be","#ffd43b","#f03e3e"]

# ── CHART 1: Benchmark de modelos ────────────────────────
# POR QUÊ barras agrupadas? Permite comparar F1-macro e ROC-AUC
# lado a lado para cada modelo — visualização direta da evolução
# da complexidade do baseline ao Gradient Boosting.
fig1 = go.Figure()
fig1.add_trace(go.Bar(
    name="F1-macro", x=bench_nomes, y=bench_f1,
    text=[f"{v:.3f}" for v in bench_f1],
    textposition="outside",
    marker_color=cores5
))
fig1.add_trace(go.Bar(
    name="ROC-AUC", x=bench_nomes, y=bench_roc,
    text=[f"{v:.3f}" for v in bench_roc],
    textposition="outside",
    marker_color=[c.replace(")", ",0.35)").replace("rgb","rgba")
                  if "rgb" in c else f"rgba(0,0,0,0.15)"
                  for c in cores5],
    marker_line_color=cores5, marker_line_width=2
))
fig1.add_shape(type="line", x0=-0.5, x1=4.5, y0=0.55, y1=0.55,
               line=dict(color="red", dash="dot", width=1.5))
fig1.add_annotation(x=4.4, y=0.57, text="Meta F1 ≥ 0.55",
                    showarrow=False, font=dict(color="red", size=11), xanchor="right")
fig1.update_layout(
    title="Benchmark de Modelos — F1-macro vs ROC-AUC",
    barmode="group",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5),
    margin=dict(t=100, b=60, l=55, r=20), height=440
)
fig1.update_xaxes(title_text="Modelo")
fig1.update_yaxes(title_text="Score", range=[0, 0.78])
# ✅ SUBSTITUTO DO KALEIDO: write_html em vez de write_image
fig1.write_html("output/charts/chart_benchmark.html")
print("✅ chart_benchmark.html salvo")

# ── CHART 2: Feature Importance ──────────────────────────
# POR QUÊ barras horizontais ordenadas? Ranking visual imediato.
# Cores por faixa de importância comunicam criticidade ao negócio.
feat_labels = {
    "saldo_bi":"Saldo (R$ bi)", "intensidade_rel":"Intensidade Rel.",
    "meses_cliente":"Meses Cliente", "retorno_relativo":"Retorno Relativo",
    "segmento_enc":"Segmento", "retorno_12m_pct":"Retorno 12m %",
    "engajamento_score":"Engajamento", "qtd_produtos":"Qtd Produtos",
    "freq_contato_mes":"Freq. Contato", "flag_risco":"Flag Risco"
}
imp = importances.copy()
imp["label"] = imp["feature"].map(feat_labels)

fig2 = go.Figure(go.Bar(
    x=imp["importance"],
    y=imp["label"],
    orientation="h",
    text=[f"{v:.3f}" for v in imp["importance"]],
    textposition="outside",
    cliponaxis=False,
    marker_color=["#f03e3e" if v > 0.15 else
                  "#ffd43b" if v > 0.08 else "#74c0fc"
                  for v in imp["importance"]],
    marker_line_color="#dee2e6", marker_line_width=1
))
fig2.update_layout(
    title="Feature Importance — Gradient Boosting (Gini)",
    margin=dict(t=80, b=55, l=125, r=80), height=420
)
fig2.update_xaxes(title_text="Importancia", range=[0, imp["importance"].max() * 1.35])
fig2.update_yaxes(title_text="")
fig2.write_html("output/charts/chart_importance.html")
print("✅ chart_importance.html salvo")

# ── CHART 3: Matriz de Confusão ───────────────────────────
# POR QUÊ heatmap? Comunica de forma visual os 4 tipos de erro
# (TN, FP, FN, TP) — muito mais intuitivo que a tabela numérica.
tn, fp, fn, tp_v = cm.ravel()
labels_cm = ["Ficou (0)", "Churnou (1)"]
fig3 = go.Figure(go.Heatmap(
    z=cm,
    x=labels_cm,
    y=labels_cm,
    text=[[str(v) for v in row] for row in cm],
    texttemplate="<b>%{text}</b>",
    textfont=dict(size=24, color="white"),
    colorscale=[[0,"#e9ecef"],[0.5,"#4dabf7"],[1,"#1971c2"]],
    showscale=False
))
fig3.update_layout(
    title=f"Matriz de Confusao — Gradient Boosting  |  TN={tn} FP={fp} FN={fn} TP={tp_v}",
    margin=dict(t=80, b=70, l=110, r=20), height=380
)
fig3.update_xaxes(title_text="Predito")
fig3.update_yaxes(title_text="Real")
fig3.write_html("output/charts/chart_confusao.html")
print("✅ chart_confusao.html salvo")

# ── CHART 4: Cross-Validation ─────────────────────────────
# POR QUÊ CV plot? Mostra que a performance não foi "sorte" do
# split — os 5 folds têm scores próximos, provando estabilidade.
fig4 = go.Figure(go.Bar(
    x=[f"Fold {i+1}" for i in range(5)],
    y=cv_scores.round(4),
    text=[f"{v:.4f}" for v in cv_scores],
    textposition="outside",
    marker_color="#74c0fc",
    marker_line_color="#4dabf7", marker_line_width=1.5
))
fig4.add_shape(type="line", x0=-0.5, x1=4.5,
               y0=cv_scores.mean(), y1=cv_scores.mean(),
               line=dict(color="#1971c2", dash="dash", width=2))
fig4.add_annotation(
    x=0, y=cv_scores.mean() + 0.005,
    text=f"Media={cv_scores.mean():.4f} ± {cv_scores.std():.4f}",
    showarrow=False, font=dict(color="#1971c2", size=11), xanchor="left")
fig4.update_layout(
    title="Cross-Validation 5-Fold — F1-macro (Gradient Boosting)",
    margin=dict(t=80, b=55, l=60, r=20), height=400
)
fig4.update_xaxes(title_text="Fold")
fig4.update_yaxes(title_text="F1-macro",
                  range=[cv_scores.min()-0.05, cv_scores.max()+0.06])
fig4.write_html("output/charts/chart_cv.html")
print("✅ chart_cv.html salvo")

# ── CHART 5: Churn por Segmento ───────────────────────────
# POR QUÊ incluir linha de média? Dá contexto imediato:
# segmentos acima da linha merecem ação prioritária.
seg_stats = (df_fe.groupby("segmento")["churn"]
               .agg(churned="sum", total="count", taxa="mean")
               .assign(taxa_pct=lambda x: (x["taxa"]*100).round(1))
               .drop(columns="taxa")
               .sort_values("taxa_pct", ascending=False)
               .reset_index())
media_geral = df_fe["churn"].mean() * 100

fig5 = go.Figure(go.Bar(
    x=seg_stats["segmento"],
    y=seg_stats["taxa_pct"],
    text=[f"{v}%" for v in seg_stats["taxa_pct"]],
    textposition="outside",
    marker_color=["#f03e3e","#ffd43b","#74c0fc","#63e6be"],
    marker_line_color="#dee2e6", marker_line_width=1
))
fig5.add_shape(type="line", x0=-0.5, x1=3.5,
               y0=media_geral, y1=media_geral,
               line=dict(color="#868e96", dash="dot", width=1.5))
fig5.add_annotation(
    x=3.4, y=media_geral + 1.5,
    text=f"Media geral: {media_geral:.1f}%",
    showarrow=False, font=dict(color="#868e96", size=11), xanchor="right")
fig5.update_layout(
    title="Taxa de Churn por Segmento  (ANOVA: F=9.4, p<0.001)",
    margin=dict(t=80, b=55, l=60, r=20), height=400
)
fig5.update_xaxes(title_text="Segmento")
fig5.update_yaxes(title_text="Churn (%)",
                  range=[0, seg_stats["taxa_pct"].max() + 8])
fig5.write_html("output/charts/chart_churn_seg.html")
print("✅ chart_churn_seg.html salvo")

print("\n✅ Todos os 5 charts salvos como HTML em output/charts/")
print("   → Abra no navegador ou use fig.show() no notebook")

✅ chart_benchmark.html salvo
✅ chart_importance.html salvo
✅ chart_confusao.html salvo
✅ chart_cv.html salvo
✅ chart_churn_seg.html salvo

✅ Todos os 5 charts salvos como HTML em output/charts/
   → Abra no navegador ou use fig.show() no notebook


In [29]:
# =============================================================
# FASE 6 – DEPLOYMENT: Persistência + Simulação de Predição
# POR QUÊ: Salvar o modelo e o scaler garante que produção
# reproduza EXATAMENTE o mesmo comportamento do treino.
# Se salvarmos só o modelo sem o scaler, os dados de entrada
# chegarão na escala errada e as predições serão inválidas.
# =============================================================
import joblib

# Salvar artefatos de produção
joblib.dump(gb_final, "output/models/gb_churn_model.pkl")
joblib.dump(scaler,   "output/models/scaler.pkl")
joblib.dump(le,       "output/models/label_encoder.pkl")

print("✅ Artefatos salvos:")
print("   output/models/gb_churn_model.pkl")
print("   output/models/scaler.pkl")
print("   output/models/label_encoder.pkl")

# ── SIMULAÇÃO DE PREDIÇÃO AO VIVO ─────────────────────────
# POR QUÊ: Demonstrar didaticamente como o sistema usaria
# o modelo em produção — dado um cliente novo, retorna score.
print("\n" + "═"*60)
print("SIMULAÇÃO DE PREDIÇÃO – 3 perfis de clientes novos")
print("═"*60)

# Recarregar artefatos (simulando ambiente de produção)
model_prod  = joblib.load("output/models/gb_churn_model.pkl")
scaler_prod = joblib.load("output/models/scaler.pkl")
le_prod     = joblib.load("output/models/label_encoder.pkl")

def prever_churn(nome, segmento, meses, qtd_prod, retorno, freq, saldo_bi):
    """
    Dado o perfil de um cliente, retorna probabilidade de churn
    e recomendação de ação para o assessor.
    """
    seg_enc = le_prod.transform([segmento])[0]
    media_ret = 11.5  # média da carteira (fixo, calculado no treino)

    # Recria features de engenharia
    engajamento    = (freq / 10) * (qtd_prod / 8)
    ret_relativo   = retorno - media_ret
    flag_risco     = int(ret_relativo < 0 and freq == 0 and qtd_prod == 1)
    intensidade    = np.log1p(meses) * np.log1p(freq)

    features = pd.DataFrame([[
        meses, qtd_prod, retorno, freq, saldo_bi,
        engajamento, ret_relativo, flag_risco, intensidade, seg_enc
    ]], columns=FEATURES)

    prob_churn = model_prod.predict_proba(features)[0][1]

    if prob_churn >= 0.60:
        acao = "🔴 ALERTA: Contato URGENTE do assessor!"
    elif prob_churn >= 0.35:
        acao = "🟡 ATENÇÃO: Agendar revisão de carteira"
    else:
        acao = "🟢 OK: Monitoramento padrão"

    print(f"\nCliente: {nome}")
    print(f"  Perfil : {segmento} | {meses} meses | {qtd_prod} produtos | "
          f"retorno {retorno}% | {freq} contatos/mês | saldo R${saldo_bi:.3f}bi")
    print(f"  Prob. Churn : {prob_churn*100:.1f}%")
    print(f"  Ação        : {acao}")
    return prob_churn

# 3 perfis distintos
p1 = prever_churn("Carlos M.", "Varejo",     8,  1, 6.2,  0, 0.012)  # alto risco
p2 = prever_churn("Ana P.",    "Alta Renda", 48, 4, 13.5, 3, 0.280)  # risco médio
p3 = prever_churn("Roberto S.","Wealth",    96, 7, 14.8, 5, 1.850)  # baixo risco

print(f"\n{'='*60}")
print("Lógica de negócio confirmada:")
print(f"  Carlos (Varejo, novo, sem contato)  → {p1*100:.1f}% churn (alto)")
print(f"  Ana    (Alta Renda, engajada)        → {p2*100:.1f}% churn (médio)")
print(f"  Roberto(Wealth, fiel, saldo alto)   → {p3*100:.1f}% churn (baixo)")

✅ Artefatos salvos:
   output/models/gb_churn_model.pkl
   output/models/scaler.pkl
   output/models/label_encoder.pkl

════════════════════════════════════════════════════════════
SIMULAÇÃO DE PREDIÇÃO – 3 perfis de clientes novos
════════════════════════════════════════════════════════════

Cliente: Carlos M.
  Perfil : Varejo | 8 meses | 1 produtos | retorno 6.2% | 0 contatos/mês | saldo R$0.012bi
  Prob. Churn : 64.4%
  Ação        : 🔴 ALERTA: Contato URGENTE do assessor!

Cliente: Ana P.
  Perfil : Alta Renda | 48 meses | 4 produtos | retorno 13.5% | 3 contatos/mês | saldo R$0.280bi
  Prob. Churn : 4.5%
  Ação        : 🟢 OK: Monitoramento padrão

Cliente: Roberto S.
  Perfil : Wealth | 96 meses | 7 produtos | retorno 14.8% | 5 contatos/mês | saldo R$1.850bi
  Prob. Churn : 3.9%
  Ação        : 🟢 OK: Monitoramento padrão

Lógica de negócio confirmada:
  Carlos (Varejo, novo, sem contato)  → 64.4% churn (alto)
  Ana    (Alta Renda, engajada)        → 4.5% churn (médio)
  Roberto(Wea